# 04 - Análise de Motivos e Perfil de GC em Sequências Longas

Este notebook reproduz as análises de interpretabilidade e perfil de GC sobre contigs longos (`long/long_loss_function.ipynb`),
fatiando contigs genômicos em janelas de 512 bp e mapeando ativações de motivos e teor de GC via `viral_cnn.interpretability.motifs`.

**Invariantes científicas (`AD-004`, `AD-014`):**
- Rastreamento de ativações de 7-mers ao longo de contigs genômicos extensos.
- Correlação com conteúdo GC local.


In [1]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from viral_cnn.notebook_runtime import setup_notebook_runtime, read_parquet_sample, reserve_c02_long_run
from viral_cnn.inference.voter import slice_sequence
from viral_cnn.m06_workflow import (
    sample_deterministic_contigs,
    load_top_motifs_for_long_analysis,
    slice_contig_and_track_motifs,
    render_figure_long_motifs_gc,
    PROTOCOL_ID,
)

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
project_root = next(r for r in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (r / "src" / "viral_cnn").exists())
run_id, output_dir = reserve_c02_long_run(project_root, "long/04")
ctx.run_id, ctx.output_dir = run_id, output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: results_reproduction/long_voter


## 2. Fatiamento em Janelas e Mapeamento de GC e Motivos de 7-mers

Fatia contigs longos em janelas de 512 bp (stride 256 bp) e computa o teor de GC e a ocorrência de motivos aprendidos.


In [2]:
# Motivos aprendidos de 1ª camada resolvidos de short/07 ou fallback representativo
motifs_tracked = load_top_motifs_for_long_analysis(
    default_motifs=["CCCCCCC", "GGGGGGG", "ATATATA", "TTTTTTT"]
)
print("Motivos 7-mers rastreados:", motifs_tracked)

if ctx.is_smoke_test:
    # Contig determinístico controlado contendo ocorrências dos 4 motivos rastreados
    sample_contig = (
        ("ACGT" * 50) + motifs_tracked[0] + 
        ("TGCA" * 50) + motifs_tracked[1] + 
        ("ACGT" * 50) + motifs_tracked[2] + 
        ("TGCA" * 50) + motifs_tracked[3] + 
        ("ACGT" * 300)
    )
    sampled_contigs_df = pd.DataFrame([{"query_id": "smoke_contig_01", "sequence": sample_contig, "length": len(sample_contig), "gc_percent": 50.0}])
else:
    long_parquet = ctx.topology.paths["long"].get("ncbi_parquet_dir")
    if long_parquet is None or not Path(long_parquet).exists():
        raise FileNotFoundError(
            f"Diretório de contigs reais do NCBI não encontrado: '{long_parquet}'. "
            "Para execução fora de smoke test, o dataset real é obrigatório."
        )
    df_long = read_parquet_sample(long_parquet, columns=["sequence", "query_id"], max_rows=None, split="test")
    if len(df_long) == 0:
        raise ValueError(f"Nenhum contig encontrado em {long_parquet}")
    sampled_contigs_df = sample_deterministic_contigs(df_long, min_length=2000, n_contigs=3, seed=42)
    sample_contig = sampled_contigs_df["sequence"].iloc[0]

print(f"Contigs amostrados deterministicamente ({len(sampled_contigs_df)} contigs):")
print(sampled_contigs_df[["query_id", "length", "gc_percent"]].to_string(index=False))

# Fatiamento em janelas de 512 bp com stride 256 bp e contagem de motivos
df_motifs = slice_contig_and_track_motifs(sample_contig, motifs_tracked, window_size=512, stride=256, handle_short="pad")
summary_csv = output_dir / "long_contig_motif_summary.csv"
df_motifs.to_csv(summary_csv, index=False)

print(f"Resumo de motivos e GC por janela de 512 bp (total de janelas: {len(df_motifs)}):")
print(df_motifs.head(10).to_string(index=False))


Resumo de motivos e GC por janela de 512 bp (total de janelas: 12):
 window_idx  start_pos  end_pos  gc_percent  motif_CCCCCCC  motif_GGGGGGG  motif_ATATATA  motif_TTTTTTT
          0          0      512       46.29              0              0              0              0
          1        256      768       47.07              0              0              0              0
          2        512     1024       48.83              0              0              0              0
          3        768     1280       49.02              0              0              0              0
          4       1024     1536       48.24              0              0              0              0
          5       1280     1792       45.90              0              0              0              0
          6       1536     2048       45.31              0              0              0              0
          7       1792     2304       44.53              0              0              0            

[LOG_PATH] Loading ParquetSample from: /home/mlima/workspace/processed_ncbi_splited_parquet/test/part.0.parquet ... (256 files total) | split=test, max_rows=1


## 3. Geração da Figura de Perfil de GC e Ocorrência de Motivos


In [3]:
fig_path = output_dir / "figure_long_motifs_gc.png"
render_figure_long_motifs_gc(
    df_motifs,
    fig_path,
    title=f"Perfil de GC e Ativação de Motivos em Sequência Longa Viral ({sampled_contigs_df['query_id'].iloc[0]})"
)
print(f"Figura gerada com sucesso: {fig_path}")


Figura gerada com sucesso: results_reproduction/long_voter/figure_long_motifs_gc.png


## 4. Emissão do Manifesto de Motivos em Contigs Longos


In [4]:
manifest_04 = {
    "notebook": "04_long_motifs_and_gc_analysis.ipynb",
    "protocol_id": PROTOCOL_ID,
    "run_id": ctx.run_id,
    "contig_query_id": str(sampled_contigs_df["query_id"].iloc[0]),
    "contig_length": len(sample_contig),
    "n_windows": len(df_motifs),
    "motifs_tracked": motifs_tracked,
    "sampled_contigs": sampled_contigs_df[["query_id", "length", "gc_percent"]].to_dict(orient="records"),
    "artifacts": {
        "summary_csv": str(summary_csv),
        "figure": str(fig_path),
    },
}

manifest_path = output_dir / "manifest_04_long_motifs.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_04, f, indent=2)

print(f"Manifesto 04 salvo em: {manifest_path}")

from viral_cnn.notebook_runtime import write_notebook_manifest
write_notebook_manifest(
    "long/04",
    ctx.data_root,
    ctx.is_smoke_test,
    metrics_summary={"n_windows": len(df_motifs), "motifs_count": len(motifs_tracked)},
    input_files=[],
    run_id=ctx.run_id,
    configuration={"protocol": PROTOCOL_ID, "motifs_tracked": motifs_tracked},
    outputs=[summary_csv, fig_path, manifest_path],
    output_json_path=output_dir / "manifest.json",
)


Manifesto 04 salvo em: results_reproduction/long_voter/manifest_04_long_motifs.json
